# Task 1: understanding the GCD pilot

This notebook reads reproducible pipeline outputs. It does not generate target labels or train a model. Our target is **arrival time**; rank values here are **BOG-derived input features**.

Generate the artifacts on the server using the commands in the README before running this notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
from rtl_timing.graph import load_bog
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/task1.json").exists())
base = ROOT / "data/processed/gcd"
assert base.exists(), "Generate GCD artifacts first"

## One circuit, four representations
The register endpoints should remain identifiable while the combinational vocabulary changes. Counts describe these representations, not physical chip area.

- **SOG:** AND, OR, NOT, XOR, and MUX operators.
- **AIG:** AND and NOT.
- **AIMG:** AND, NOT, and MUX.
- **XAG:** AND, NOT, and XOR.

Buffers are allowed by the restricted libraries. A richer vocabulary can express the same logic with fewer operators. Comparing the four views gives a model different clues about logic complexity; none is the final target circuit. Formal checks test whether mapping preserved behavior.

In [2]:
summaries = [json.loads(p.read_text()) for p in sorted(base.glob("*/summary.json"))]
pd.DataFrame(summaries)[["library_variant", "representation", "register_bits", "combinational_operators", "max_combinational_depth", "timed_endpoints", "valid_sampled_paths"]]

,library_variant,representation,register_bits,combinational_operators,max_combinational_depth,timed_endpoints,valid_sampled_paths
0,reset_v1,aig,34,684,62,34,1088
1,reset_v1,aimg,34,555,62,34,951
2,reset_v1,sog,34,290,37,34,920
3,reset_v1,xag,34,579,61,34,1088


## Inspect one endpoint
The aliases connect a mapped register back to named RTL wires. Its input cone contains the logic that can drive its D input; traversal stops at source registers and primary inputs.

In [3]:
bog = load_bog(base / "sog/bog.json", "gcd", "sog")
endpoint = next(bog.endpoints())
registers, inputs, operators = bog.cone(endpoint["d_bit"])
{"endpoint": endpoint, "driving_registers": len(registers), "primary_inputs": len(inputs), "cone_operators": len(operators)}

{'endpoint': {'cell': 'ctrl.state.out[0]$_SDFF_PP0_',
  'd_bit': 206,
  'q_bit': 56,
  'aliases': ['ctrl.curr_state__0[0]',
   'ctrl.current_state__1[0]',
   'ctrl.state$out[0]',
   'ctrl.state.out[0]'],
  'endpoint_id': 'ctrl.curr_state__0[0]'},
 'driving_registers': 34,
 'primary_inputs': 2,
 'cone_operators': 86}

## Inspect paths and arrival times
The critical row is selected from separate rising/falling timing queries. Sampled rows represent additional graph routes verified against OpenSTA. Times are ns; capacitances are fF. These are **BOG timing features**, not Task 2 labels.

In [4]:
paths = pd.read_parquet(base / "sog/path_features.parquet")
rows = paths[paths.endpoint_cell == endpoint["cell"]]
rows[["path_kind", "edge", "bog_arrival_ns", "path_depth", "operator_and_count", "operator_xor_count", "fanout_mean", "capacitance_ff_mean", "slew_ns_mean"]].head(12)

,path_kind,edge,bog_arrival_ns,path_depth,operator_and_count,operator_xor_count,fanout_mean,capacitance_ff_mean,slew_ns_mean
0,critical,rise,1.382680,34,16,1,1.600000,2.064535,0.008031
1,sampled,fall,0.227345,5,2,0,2.166667,2.341291,0.008423
2,sampled,rise,0.209854,5,2,0,1.833333,2.049711,0.008341
3,sampled,fall,0.179332,3,2,0,1.500000,1.496177,0.006443
4,sampled,fall,0.382542,7,1,0,1.375000,1.303453,0.007667
5,sampled,fall,0.340791,6,1,0,1.428571,1.376462,0.007614
6,sampled,fall,0.335577,6,1,0,1.428571,1.361739,0.007598
7,sampled,fall,0.340789,6,1,0,1.428571,1.376462,0.007614
8,sampled,fall,0.387756,7,1,0,1.375000,1.316335,0.007681
9,sampled,fall,0.335713,6,1,0,1.428571,1.361739,0.007602


## A numerical consistency check
No sampled route should exceed the maximum-arrival path for the same endpoint under the same constraints.

In [5]:
critical = paths[paths.path_kind == "critical"].set_index("endpoint_cell").bog_arrival_ns
maximum = paths.groupby("endpoint_cell").bog_arrival_ns.max()
assert ((maximum - critical) <= 1e-7).all()
{"checked_endpoints": len(critical), "result": "passed"}

{'checked_endpoints': 34, 'result': 'passed'}

## Verification and limits
Formal checks compare mapped BOG logic to the pre-mapping Yosys elaboration. They do not establish original-RTL frontend correctness. The current pilot uses the harmonized common SDC (10 ns clock, zero latency, 0.1 ns I/O delay), ideal clocks, no extracted wire parasitics, and a typical Nangate45 library. Read `docs/task1-features.md` for deviations, including statistical aggregation and path sampling.

The current collection uses `reset_v1`: the original restricted libraries plus `DFFR_X1` and `DFFS_X1`. All original combinational cells and the proof method are unchanged.

In [6]:
{p.parent.name: json.loads(p.read_text()) for p in sorted(base.glob("*/equivalence.json"))}

{'aig': {'status': 'passed',
  'scope': 'mapped BOG versus premap elaboration; not original RTL frontend equivalence',
  'proof_semantics': 'inductive equivalence after state alignment; not reset-sequence reachability'},
 'aimg': {'status': 'passed',
  'scope': 'mapped BOG versus premap elaboration; not original RTL frontend equivalence',
  'proof_semantics': 'inductive equivalence after state alignment; not reset-sequence reachability'},
 'sog': {'status': 'passed',
  'scope': 'mapped BOG versus premap elaboration; not original RTL frontend equivalence',
  'proof_semantics': 'inductive equivalence after state alignment; not reset-sequence reachability'},
 'xag': {'status': 'passed',
  'scope': 'mapped BOG versus premap elaboration; not original RTL frontend equivalence',
  'proof_semantics': 'inductive equivalence after state alignment; not reset-sequence reachability'}}

## From this example to the collection

The inventory below reads measured artifact counts, not estimates from lines of RTL. Designs sharing a family must stay together in the later train/test split. An `unproven` result is an unresolved formal check, even when feature extraction succeeded.

In [7]:
inventory = pd.read_csv(ROOT / "docs/results/task1_inventory.csv")
inventory[inventory.representation == "sog"][["design", "family", "register_bits", "combinational_operators", "timed_endpoints", "equivalence"]]

,design,family,register_bits,combinational_operators,timed_endpoints,equivalence
0,gcd,orfs_gcd,34,290,34,passed
4,timer32,chameleon_peripherals,65,587,65,passed
8,pwm32,chameleon_peripherals,65,653,65,passed
12,wdt32,chameleon_peripherals,34,318,34,passed
16,eth_crc,opencores_ethernet,32,149,32,passed
20,eth_random,opencores_ethernet,20,114,20,passed
24,eth_txstatem,opencores_ethernet,12,243,12,passed
28,eth_txcounters,opencores_ethernet,35,503,35,passed
32,eth_rxcounters,opencores_ethernet,25,403,25,passed
36,eth_rxaddrcheck,opencores_ethernet,4,380,4,passed
